# MLP on MNIST — From Scratch using NumPy
**Target:** 98%+ Test Accuracy  
**Architecture:** 784 → 256 → 128 → 10  
**Optimizer:** Adam  
**Only allowed libraries:** `numpy`, `nnfs`, `sklearn` (for data loading only)

In [ ]:
!pip install nnfs

## 1. Imports & Setup

In [ ]:
import numpy as np
import nnfs
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

np.random.seed(42)
nnfs.init()

## 2. Load & Preprocess MNIST

In [ ]:
def load_clean_mnist():
    print("Downloading MNIST (may take a minute)...")
    mnist = fetch_openml('mnist_784', version=1, cache=True, parser='auto')
    X = np.array(mnist.data, dtype=np.float64)
    y = np.array(mnist.target, dtype=int)
    X = X / 255.0   # normalize pixel values to [0, 1]
    return X, y

X, y = load_clean_mnist()
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=10000, random_state=42)
print(f"\nX_train (Training Images) Shape: {X_train.shape}")
print(f"y_train (Training Labels) Shape: {y_train.shape}")
print(f"X_test  (Testing Images)  Shape: {X_test.shape}")
print(f"y_test  (Testing Labels)  Shape: {y_test.shape}")

## 3. Network Components

### DenseLayer
Supports **He initialization** for ReLU networks and **Adam** optimizer built-in.

In [ ]:
class DenseLayer:
    def __init__(self, n_inputs, n_neurons):
        # He initialization — critical for deep ReLU networks to avoid vanishing gradients
        self.weights = np.random.randn(n_inputs, n_neurons) * np.sqrt(2.0 / n_inputs)
        self.biases  = np.zeros((1, n_neurons))
        # Adam moment estimates (first and second moment vectors)
        self.mw = np.zeros_like(self.weights)
        self.vw = np.zeros_like(self.weights)
        self.mb = np.zeros_like(self.biases)
        self.vb = np.zeros_like(self.biases)
        self.t  = 0  # timestep counter for bias correction

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    def calc_gradient(self, output_gradient):
        self.weights_gradient = np.dot(self.inputs.T, output_gradient)
        self.biases_gradient  = np.sum(output_gradient, axis=0, keepdims=True)
        self.input_gradient   = np.dot(output_gradient, self.weights.T)

    def adam_update(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.t += 1
        # Weights — compute biased moment estimates, then correct them
        self.mw = beta1 * self.mw + (1 - beta1) * self.weights_gradient
        self.vw = beta2 * self.vw + (1 - beta2) * (self.weights_gradient ** 2)
        mw_hat  = self.mw / (1 - beta1 ** self.t)
        vw_hat  = self.vw / (1 - beta2 ** self.t)
        self.weights -= lr * mw_hat / (np.sqrt(vw_hat) + eps)
        # Biases
        self.mb = beta1 * self.mb + (1 - beta1) * self.biases_gradient
        self.vb = beta2 * self.vb + (1 - beta2) * (self.biases_gradient ** 2)
        mb_hat  = self.mb / (1 - beta1 ** self.t)
        vb_hat  = self.vb / (1 - beta2 ** self.t)
        self.biases -= lr * mb_hat / (np.sqrt(vb_hat) + eps)

### Activation Functions

In [ ]:
class Activation_ReLU:
    def forward(self, inputs):
        self.output = np.maximum(0, inputs)

    def calc_gradient(self, output_gradient):
        self.input_gradient = output_gradient.copy()
        self.input_gradient[self.output <= 0] = 0


class Softmax_CCE:
    """Fused Softmax + Cross-Entropy for a numerically stable backward pass."""
    def forward(self, inputs):
        # Subtract max for numerical stability before exponentiation
        exp_values  = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        self.output = exp_values / np.sum(exp_values, axis=1, keepdims=True)

    def calc_gradient(self, output_gradient, y_true):
        samples = len(output_gradient)
        self.input_gradient = output_gradient.copy()
        self.input_gradient[range(samples), y_true] -= 1
        self.input_gradient = self.input_gradient / samples


class CCE:
    """Categorical Cross-Entropy Loss"""
    def forward(self, y_pred, y_true):
        samples     = len(y_pred)
        y_pred_clip = np.clip(y_pred, 1e-7, 1 - 1e-7)
        if y_true.ndim == 1:
            correct = y_pred_clip[range(samples), y_true]
        else:
            correct = np.sum(y_pred_clip * y_true, axis=1)
        return np.mean(-np.log(correct))

## 4. Initialize Network

**Architecture choices and why they achieve 98%+:**
- **784 → 256 → 128 → 10**: Large hidden layers give the network enough capacity to learn complex digit features
- **He initialization**: Prevents vanishing/exploding gradients in ReLU networks  
- **Adam optimizer**: Adaptive learning rates per parameter — converges much faster and better than plain SGD  
- **Epoch shuffling**: Prevents the network from memorizing the order of training data

In [ ]:
# Architecture: 784 → 256 → 128 → 10
HL1       = DenseLayer(784, 256)
act1      = Activation_ReLU()
HL2       = DenseLayer(256, 128)
act2      = Activation_ReLU()
out_layer = DenseLayer(128, 10)
softmax   = Softmax_CCE()
loss_fn   = CCE()

print("Network initialized successfully!")
print("Architecture: 784 → 256 (ReLU) → 128 (ReLU) → 10 (Softmax)")

## 5. Training Loop

In [ ]:
lr              = 0.001
epochs          = 25
batch_size      = 128
steps_per_epoch = len(X_train) // batch_size

print("=" * 55)
print("TRAINING")
print("=" * 55)

for epoch in range(epochs):
    # Shuffle training data at the start of every epoch
    indices          = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[indices]
    y_train_shuffled = y_train[indices]

    epoch_loss = 0.0
    epoch_acc  = 0.0

    for step in range(steps_per_epoch):
        s, e = step * batch_size, (step + 1) * batch_size
        Xb   = X_train_shuffled[s:e]
        yb   = y_train_shuffled[s:e]

        # ── Forward Pass ──
        HL1.forward(Xb);           act1.forward(HL1.output)
        HL2.forward(act1.output);  act2.forward(HL2.output)
        out_layer.forward(act2.output)
        softmax.forward(out_layer.output)

        # ── Loss & Accuracy ──
        epoch_loss += loss_fn.forward(softmax.output, yb)
        epoch_acc  += np.mean(np.argmax(softmax.output, axis=1) == yb)

        # ── Backward Pass ──
        softmax.calc_gradient(softmax.output, yb)
        out_layer.calc_gradient(softmax.input_gradient)
        act2.calc_gradient(out_layer.input_gradient)
        HL2.calc_gradient(act2.input_gradient)
        act1.calc_gradient(HL2.input_gradient)
        HL1.calc_gradient(act1.input_gradient)

        # ── Adam Weight Update ──
        HL1.adam_update(lr)
        HL2.adam_update(lr)
        out_layer.adam_update(lr)

    avg_acc  = epoch_acc  / steps_per_epoch
    avg_loss = epoch_loss / steps_per_epoch
    print(f"Epoch {epoch+1:2d}/{epochs} | Train Acc: {avg_acc:.4f} | Loss: {avg_loss:.4f}")

## 6. Final Test Evaluation

In [ ]:
# Forward pass on UNSEEN test data
HL1.forward(X_test);           act1.forward(HL1.output)
HL2.forward(act1.output);      act2.forward(HL2.output)
out_layer.forward(act2.output); softmax.forward(out_layer.output)

test_predictions = np.argmax(softmax.output, axis=1)
test_accuracy    = np.mean(test_predictions == y_test)
test_loss        = loss_fn.forward(softmax.output, y_test)

print("=" * 55)
print("TRAINING COMPLETE. COMMENCING FINAL EXAM...")
print("=" * 55)
print(f"FINAL TEST ACCURACY: {test_accuracy * 100:.2f}%")
print(f"FINAL TEST LOSS:     {test_loss:.4f}")
print("=" * 55)